# CFTC 2025 积压报告：计划发布时间与周报更新日期
仅核对两个冻结报告日和 CFTC 官方公告表，不读市场结果。

In [1]:
import datetime,hashlib,json,re,shlex,subprocess,tarfile,tempfile
from pathlib import Path
from bs4 import BeautifulSoup
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd().resolve();card=root/'research/stage1_cftc_tff_2025_backlog_update_audit_20261023';plan=json.loads((card/'plan.json').read_text());receipt=json.loads((card/'capture_receipts.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and receipt['plan_sha256']==sha(card/'plan.json') and receipt['GETs_attempted']==3
paths=[card/'plan.json',card/'capture_receipts.json'];soups=[]
for i,row in enumerate(receipt['captures']):
 assert row['index']==i and row['url']==plan['exact_urls'][i] and row['curl_exit']==0 and row['http_status']=='200'
 for kind in ('body','headers'):
  entry=row[kind];path=root/entry['path'];assert path.stat().st_size==entry['bytes'] and sha(path)==entry['sha256'];paths.append(path)
 soups.append(BeautifulSoup((root/row['body']['path']).read_bytes(),'html.parser'))
schedule=[]
for table in soups[0].find_all('table'):
 rows=[]
 for tr in table.find_all('tr'):
  cells=[x.get_text(' ',strip=True) for x in tr.find_all(['th','td'])]
  if cells:rows.append(cells)
 if any(x[0]=='12/23/2025' for x in rows):schedule.append(rows)
assert len(schedule)>=2
final_schedule=schedule[0];checks=[]
for case in plan['cases']:
 rows=[x for x in final_schedule if x[0]==datetime.date.fromisoformat(case['report_date']).strftime('%m/%d/%Y')]
 assert len(rows)==1
 scheduled=rows[0][2].replace('+','');assert scheduled==datetime.date.fromisoformat(case['scheduled_publish_date']).strftime('%m/%d/%Y')
 text=soups[case['weekly_url_index']].get_text(' ',strip=True)
 report=datetime.date.fromisoformat(case['report_date']);assert ('Positions as of '+report.strftime('%B ')+str(report.day)+', '+str(report.year)) in text
 dates=re.findall(r'Updated\s+([A-Za-z]+\s+\d{1,2},\s+\d{4})',text);assert len(dates)==1,dates
 updated=datetime.datetime.strptime(dates[0],'%B %d, %Y').date();scheduled_date=datetime.date.fromisoformat(case['scheduled_publish_date']);planned_available=scheduled_date+datetime.timedelta(days=1)
 checks.append({'report_date':case['report_date'],'scheduled_publish_date':case['scheduled_publish_date'],'weekly_page_updated_date':updated.isoformat(),'planned_next_day_available':planned_available.isoformat(),'updated_after_planned_available':updated>planned_available,'weekly_body_sha256':receipt['captures'][case['weekly_url_index']]['body']['sha256'],'first_public_time_proved':False})
assert checks[0]['updated_after_planned_available'] is False and checks[1]['updated_after_planned_available'] is True
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='cftc-backlog-audit-') as directory:
 archive=Path(directory)/'raw.tar.gz'
 with tarfile.open(archive,'w:gz') as output:
  for path in paths:output.add(path,arcname=str(path.relative_to(root)),recursive=False)
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote_path)).split()[0]==digest
 restored=Path(directory)/'restored.tar.gz'
 with restored.open('wb') as out:transfer=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote_path)],stdout=out,stderr=subprocess.PIPE,timeout=60)
 assert transfer.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as source:
  members={m.name:m for m in source.getmembers()};assert set(members)=={str(p.relative_to(root)) for p in paths}
  for path in paths:assert source.extractfile(members[str(path.relative_to(root))]).read()==path.read_bytes()
 evidence['nas_path']=remote_path
result={'status':'2025_12_23_current_weekly_version_updated_after_planned_availability','plan_sha256':sha(card/'plan.json'),'GETs_consumed':3,'checks':checks,'scientific_interpretation':'Updated is not first-public proof, but the currently served 2025-12-23 weekly report version is marked December 31 after a December 29 planned release and December 30 proposed Stage1 availability. Current annual values at December 30 are therefore unqualified without a first-release snapshot; mark unknown or use a later conservative date after independently verifying the version.','NAS_evidence':evidence,'NAS_remote_and_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':card.name,'decision':'needs_timing_repair_2025_12_23_backlog_current_version_after_planned_availability','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','GETs_consumed':3,'NAS_archive_sha256':evidence['sha256'],'NAS_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(summary['decision'],checks,evidence['sha256'])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


needs_timing_repair_2025_12_23_backlog_current_version_after_planned_availability [{'report_date': '2025-12-02', 'scheduled_publish_date': '2025-12-17', 'weekly_page_updated_date': '2025-12-17', 'planned_next_day_available': '2025-12-18', 'updated_after_planned_available': False, 'weekly_body_sha256': '36a3a2fe2a9ae54a2a75c33e4cb6f0f8d6459053506edda6fea5f89b017bd964', 'first_public_time_proved': False}, {'report_date': '2025-12-23', 'scheduled_publish_date': '2025-12-29', 'weekly_page_updated_date': '2025-12-31', 'planned_next_day_available': '2025-12-30', 'updated_after_planned_available': True, 'weekly_body_sha256': '4561adb75cc0be50de3ac7e42893697da45c60e98b7ffab3384b4b449f591dde', 'first_public_time_proved': False}] 659a15228216d97e2438735137943d7ca35f4e53720e0617c248e96fda258188


## 中文结论
# CFTC 2025 积压报告：计划发布与当前周报更新时点

固定三份 CFTC 官方页面（12 月 9 日特殊公告、2025-12-02 和 2025-12-23 两期 TFF Futures Only 周报），各请求一次，原始正文/响应头、UTC 时刻和哈希已在本机与 Yanxuan NAS 留存，远端哈希及限定逐文件恢复通过。响应头含临时 Cookie，原字节不提交 Git。0 市场收益读取、0 拟合、0 账户。

12 月 9 日公告表把 12-02 报告安排在 12-17 发布，其当前周报页标记 `Updated December 17, 2025`。12-23 报告安排 12-29 发布，但当前页标记 `Updated December 31, 2025`。`Updated` 不是首次公开证明；它说明当前页面版本晚于计划 12-30 的 Stage1 可用时点，无法认定当前年度文件中的该行在 12-30 已同值公开。

对已封存的 Qlib 14 日映射做一次只读定向核验：2025-12-30 信号确实选用了 12-23 报告，`source_known=true`，可用时点写为 12-30 00:00 美东。因此现有映射**在进入任何信息拟合前必须修复时点**；保存原计划与 Recorder，另冻结修复版，将该报告和依赖它的 delta4 行设未知，除非取得精确首发版本。不能凭本卡断言全历史发布日、因子有效或 SOXX 超额。

证据：[CFTC 特殊公告](https://www.cftc.gov/MarketReports/CommitmentsofTraders/HistoricalSpecialAnnouncements/index.htm)、官方 [12-23 TFF 周报](https://www.cftc.gov/sites/default/files/files/dea/cotarchives/2025/futures/financial_lf122325.htm)。
